# Collaborative filtering — audited revision

Original 02B is preserved. This version adds diagnostics and a retrospective 2018 development comparison. Read the scope and candidate/neighbour definitions before interpreting scores.

## 1. Why use other travellers’ records?

Category matching cannot distinguish many places that share broad labels. Here we ask whether people who liked the same specific places can suggest additional choices. This is user-based collaborative filtering. We reuse the historical preparation rules and destination from 02A, but this notebook reconstructs its inputs independently. It does not run 02A or train a Transformer.

Inputs: prepared review arrays and catalogue. Outputs: an isolated `runs/*_cf_*` folder. Sections show a hand example, real neighbours, score contributions, list sources, sensitivity and operational checks. Shared ranking-quality evaluation remains undecided.

**Audited revision:** the original 02B remains preserved. This revision adds raw-data provenance, full sensitivity traces, targeted counterexamples and a fixed 2018 retrospective development comparison. It does not choose parameters or claim a historically exact deployment test: attraction categories have no annual versions. Candidate means a place under consideration; neighbour means another person with overlapping likes, not a nearby resident.

Run from the first cell with the persistent Tourism kernel. No variables or outputs from 02A or the old 02B are needed. Prepared files must exist under `Technical_Lab/assets/baseline_data`; original attr/reviews live privately under `assets/raw_incultum`. Download from the original INCULTUM source and follow the case setup instructions; this is not a standalone learner package.

**What this file reproduces:** the local fresh-kernel run from the prepared inputs. The full raw attr/review reconciliation is run separately with `source/audit_cf_raw_data.py` and linked by hash. Regenerating prepared inputs on a new machine still requires the existing preparation procedure and explicit raw path configuration; hosted Colab and a standalone learner package are not validated here.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, hashlib, uuid, os
import numpy as np
import pandas as pd
from IPython.display import display


# Find the Tourism folder by looking for its prepared data files.
LAB = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'assets/baseline_data').exists()))
DATA = LAB / 'assets/baseline_data'


# Give this run its own folder so earlier results are not overwritten.
OUT = LAB / 'Current/interactive/runs' / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_cf_audited_' + uuid.uuid4().hex[:6])
OUT.mkdir(parents=True)


# Choose the history cutoff, destination, minimum reviewer count and list length here.
CONFIG = dict(train_before='2018-01-01', city='France/Paris', min_item_positive_users=5, top_k=10)


# Save the settings beside the results so we know how this list was made.
(OUT / 'configuration.json').write_text(json.dumps(CONFIG, indent=2))
print('Output:', OUT)
print(
    'Candidate support 5 is retained from the earlier experiment; it is adjustable, not a proven optimum.',
)


# Record the actual environment and freeze the comparison plan before reading any later labels.
import sys, platform, importlib.metadata, time
AUDIT_STARTED = time.perf_counter()
versions = {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'scipy', 'nbformat', 'nbclient', 'ipykernel']}
(OUT / 'environment.json').write_text(
    json.dumps(
        {'python': sys.version, 'platform': platform.platform(), 'packages': versions},
        indent=2,
    ),
)
protocol = json.loads(
    (LAB.parent / 'Reviews/Notebook_Audit_2026-10-17/Development_Comparison_Protocol.json').read_text(),
)
(OUT / 'comparison_protocol_before_labels.json').write_text(json.dumps(protocol, indent=2))
print(
    'Demonstration: pre-2018 history. Later step: retrospective 2018 development comparison, not final testing.',
)


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 2. A small interaction matrix

Illustration: each row is a person, each column a specific place. A 1 means an observed 4/5 rating; 0 means no observed like, not a dislike. These columns are attraction identities, not the 20 category labels from 02A.

In [ ]:
# Make an invented person-by-place table. A 1 is an observed like; a 0 is not proof of dislike.
toy = pd.DataFrame(
    [[1, 1, 0, 0], [1, 1, 1, 0], [1, 0, 0, 1], [0, 0, 1, 1]],
    index=['Query', 'Neighbour A', 'Neighbour B', 'Other'],
    columns=['Place A', 'Place B', 'Place C', 'Place D'],
)
display(toy)


# Convert the visible table to numbers and calculate each person’s vector length.
a = toy.to_numpy(dtype=float)
norms = np.linalg.norm(a, axis=1)


# Compare each person with the query, then remove the query’s self-similarity.
sim = a @ a[0] / (norms * norms[0])
sim[0] = 0


# Only people with positive shared-like similarity may vote in this example.
toy_neighbours = np.flatnonzero(sim > 0)


# Weight each neighbour’s likes by similarity and divide by all selected neighbour weights.
toy_scores = a[toy_neighbours].T @ sim[toy_neighbours] / sim[toy_neighbours].sum()
display(pd.DataFrame({'person': toy.index, 'cosine_to_query': sim}))
display(pd.DataFrame({'candidate': ['Place C', 'Place D'], 'weighted_vote': toy_scores[2:]}))


# Check the similarity and candidate scores against the hand calculation.
assert np.isclose(sim[1], 2 / np.sqrt(6)) and np.isclose(sim[2], 0.5)


# Check the similarity and candidate scores against the hand calculation.
assert np.isclose(toy_scores[2], 2 / np.sqrt(6) / (2 / np.sqrt(6) + 0.5))


# Save this invented example separately from the real neighbour results.
toy.to_csv(OUT / 'illustration_interactions.csv')
pd.DataFrame({'similarity': sim}, index=toy.index).to_csv(OUT / 'illustration_similarities.csv')


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 3. Rebuild the historical records

Use the same deduplication, 4/5 positive ratings and pre-2018 cutoff as 02A. All earlier reviews exclude a place from repeat recommendations. This preparation step reads real data; the small example above was invented.

The separate full prepared-data audit found 122,229 repeated rows in the four model fields (person, place, publication day, stored rating), and 262,085 extra person–place rows before model deduplication. These are different counts. The notebook deduplicates preference labels before building binary likes. [Detailed checks](../../../Reviews/Notebook_Audit_2026-10-17/Prepared_Data_Quality.json) also execute the actual dedup code on low-rating, conflicting-rating, duplicate, row-order and future-date counterexamples. Equality of all 15 raw review fields was not checked because those unused fields do not enter this method.

In [ ]:
# Read a file in small pieces and calculate its fingerprint, without loading the whole file into memory.
def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


# Check that these prepared files are the same ones listed in the preparation record.
receipt = json.loads((DATA / 'preparation_receipt.json').read_text())
actual = {name: sha(DATA / name) for name in ['items.csv', 'categories.npy', 'users.npy', 'events.npy']}
assert actual == receipt['sha256_derived']
(OUT / 'input_hashes.json').write_text(json.dumps(actual, indent=2))


# Load attraction details, category labels and review records. Read the large review array from disk as needed.
items = pd.read_csv(DATA / 'items.csv', keep_default_na=False)
categories = np.load(DATA / 'categories.npy').astype(float)
events = np.load(DATA / 'events.npy', mmap_mode='r')


# Keep the row counts and category names together: array row numbers must refer to the same people and places.
NITEM = len(items)
NUSER = len(np.load(DATA / 'users.npy', mmap_mode='r'))
category_names = list(items.columns[4:])


# Stop if the attraction rows or the 20 category columns do not line up, or an attraction ID appears twice.
assert categories.shape == (NITEM, 20) and len(category_names) == 20 and items.attractionID.is_unique
assert np.isin(categories, [0, 1]).all()


# Make every nonempty attraction vector length one. Leave an empty category vector at zero.
unit_categories = categories / np.maximum(np.linalg.norm(categories, axis=1, keepdims=True), 1)


# Convert the cutoff to the same day-number format used in the review file.
cut = int(np.datetime64(CONFIG['train_before'], 'D').astype(np.int64))
prior = events['day'] < cut


# Remember every earlier reviewed place, even when its rating is low or invalid, so we do not suggest it again.
seen_pairs = np.unique(events['user'][prior].astype(np.int64) * NITEM + events['item'][prior])


# Only ratings 1 through 5 can describe a preference. A stored zero is not a dislike.
valid = np.flatnonzero((events['rating'] >= 1) & (events['rating'] <= 5))


# Give each person–place combination one numeric key: person row × number of attractions + attraction row.
pair = events['user'][valid].astype(np.int64) * NITEM + events['item'][valid]


# Put each person–place group together, with its earliest date first; rating breaks same-day ties.
order = np.lexsort((events['rating'][valid], events['day'][valid], pair))
ordered = valid[order]
keys = pair[order]


# Find where each new person–place group begins, then take its earliest valid review.
starts = np.r_[0, np.flatnonzero(np.diff(keys)) + 1]
first = ordered[starts]


# Check all ratings on that earliest day, not just the first row. Ignore later-day ratings for this conflict check.
same_day = events['day'][ordered] == np.repeat(events['day'][first], np.diff(np.r_[starts, len(keys)]))
ratings = events['rating'][ordered]


# If the smallest and largest first-day ratings differ, leave this pair out instead of choosing one rating.
conflict = np.minimum.reduceat(np.where(same_day, ratings, 127), starts) != np.maximum.reduceat(np.where(same_day, ratings, -1), starts)


# Keep one non-conflicting earliest rating per person and place, then keep only records before the cutoff.
dedup = np.array(events[first[~conflict]])
train = dedup[dedup['day'] < cut]
positive = train[train['rating'] >= 4]


# Count different people who gave each place 4 or 5. Deduplication above makes each person count once.
support = np.bincount(positive['item'], minlength=NITEM)


# For category interests, a liked place also needs a category label. CF can still use likes without labels.
usable = positive[categories[positive['item']].sum(axis=1) > 0]
history_count = np.bincount(usable['user'], minlength=NUSER)


# Keep the record counts and people count separate when showing what survived preparation.
prep = {
    'prepared_event_rows': len(events),
    'training_valid_pairs': len(train),
    'training_positive_pairs': len(positive),
    'training_usable_category_positive_pairs': len(usable),
    'all_period_conflicting_pairs': int(conflict.sum()),
    'people_with_usable_profile': int((history_count > 0).sum()),
}
display(pd.Series(prep).to_frame('count'))
(OUT / 'preparation_counts.json').write_text(json.dumps(prep, indent=2))


# Free the large temporary sorting arrays; keep the history and candidate inputs needed below.
del valid, pair, order, ordered, keys, starts, first, same_day, ratings


# Check all compact IDs, dates and rating values before using them as array positions.
assert events.dtype.names == ('user', 'item', 'day', 'rating')
assert events['user'].min() >= 0 and events['user'].max() < NUSER
assert events['item'].min() >= 0 and events['item'].max() < NITEM
assert np.isin(events['rating'], [0, 1, 2, 3, 4, 5]).all()
assert (train['day'] < cut).all() and np.isin(positive['rating'], [4, 5]).all()
train_keys = train['user'].astype(np.int64) * NITEM + train['item']
assert len(np.unique(train_keys)) == len(train)
assert np.array_equal(support, np.bincount(positive['item'], minlength=NITEM))


# Display rating counts and the distribution of historical likes, keeping records and people separate.
rating_counts = pd.Series(events['rating']).value_counts().sort_index().rename_axis('stored_rating').to_frame(
    'records',
)
activity = pd.DataFrame(
    {
        'positive_history_group': pd.cut(
            np.bincount(positive['user'], minlength=NUSER),
            [-1, 0, 2, 5, 10, np.inf],
            labels=['0', '1-2', '3-5', '6-10', '11+'],
        ),
    },
)
activity_summary = activity.value_counts(sort=False).rename('people').reset_index()
display(rating_counts)
display(activity_summary)
rating_counts.to_csv(OUT / 'prepared_rating_counts.csv')
activity_summary.to_csv(OUT / 'global_history_groups.csv', index=False)


# The raw audit is a separate full scan, not an old result disguised as a new calculation.
raw_audit_path = LAB.parent / 'Reviews/Notebook_Audit_2026-10-17/Raw_Data_Reconciliation.json'
if raw_audit_path.exists():
    raw_audit = json.loads(raw_audit_path.read_text())
    assert raw_audit['all_retained_compact_event_fields_match_raw_in_order']
    (OUT / 'raw_audit_reference.json').write_text(
        json.dumps(
            {
                'path': str(raw_audit_path),
                'sha256': sha(raw_audit_path),
                'status': raw_audit['status'],
            },
            indent=2,
        ),
    )
    display(pd.Series(raw_audit['counts']).to_frame('full_raw_scan_count'))
else:
    print(
        'Raw reconciliation NOT VERIFIED in this run; execute source/audit_cf_raw_data.py with the real raw files.',
    )


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 4. Choose Paris and anchor the same real person

A visitor selects Paris. Keep places with at least five different pre-2018 positive reviewers and at least one category. The category condition is retained so all fallback routes share a candidate set; CF itself does not require category labels. This is a discovery list, not verified opening hours or precise navigation.

In [ ]:
# Keep only places in the visitor’s chosen city; city is not a cosine feature.
city = items.locationAttraction.eq(CONFIG['city']).to_numpy()
has_category = categories.sum(axis=1) > 0


# A place enters this list only if it is in the city, has enough different high-rating reviewers, and has a category.
candidate_mask = city & (support >= CONFIG['min_item_positive_users']) & has_category
candidates = np.flatnonzero(candidate_mask)


# Show how many places each rule removes. These count labels describe the current five-reviewer example.
audit = pd.DataFrame(
    {
        'stage': [
            'Exact city',
            f"City + >={CONFIG['min_item_positive_users']} positive reviewers",
            f"City + >={CONFIG['min_item_positive_users']} + category",
        ],
        'attractions': [
            int(city.sum()),
            int((city & (support >= CONFIG['min_item_positive_users'])).sum()),
            len(candidates),
        ],
    },
)
display(audit)


# Show which category labels are available among the city’s attractions.
display(
    pd.DataFrame(
        {
            'feature': category_names,
            'city_attractions_with_label': categories[city].sum(axis=0).astype(int),
        },
    ),
)


# Save the filtering counts so the candidate choice can be checked later.
audit.to_csv(OUT / 'candidate_filters.csv', index=False)
print(
    'Category labels describe the content fallback. CF uses liked attraction IDs; city and reviewer counts set eligibility.',
)


# Find people with a usable earlier like at this destination, without looking at later reviews.
local_users = np.unique(usable['user'][candidate_mask[usable['item']]])


# Pick a fixed person with at least three likes so we can inspect several records. One like is enough to run the method.
demo_options = local_users[history_count[local_users] >= 3]
if len(demo_options) == 0:
    raise ValueError('No walkthrough person has three usable earlier likes in this destination. Choose another city or explicitly supply a valid demo user; no recommendations have been produced.')
DEMO_USER = int(demo_options[0])


# Get this person’s distinct liked places, including places in other cities.
liked = usable['item'][usable['user'] == DEMO_USER]


# Average their unit-length attraction vectors, then make the average length one for cosine scoring.
raw_profile = unit_categories[liked].mean(axis=0)
profile = raw_profile / np.linalg.norm(raw_profile)


# Show the actual liked places and supplied labels so we can explain where the interest weights came from.
history = items.iloc[liked][['attractionID', 'nameAttraction', 'locationAttraction']].copy()
history['category_labels'] = ['; '.join(np.array(category_names)[categories[i] > 0]) for i in liked]
history['rating'] = usable['rating'][usable['user'] == DEMO_USER]
display(history)


# Show the mean before final normalization and the finished weight for each category.
profile_table = pd.DataFrame(
    {
        'category': category_names,
        'average_normalized_place_weight': raw_profile,
        'final_interest_weight': profile,
    },
).sort_values(
    'final_interest_weight',
    ascending=False,
)
display(profile_table[profile_table.final_interest_weight > 0])
print('Anonymous compact user index:', DEMO_USER, '; distinct usable liked places:', len(liked))


# Save both the history and the profile so the explanation can be traced back to records.
history.to_csv(OUT / 'real_history.csv', index=False)
profile_table.to_csv(OUT / 'real_profile.csv', index=False)


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 5. Build a sparse person–place matrix

Sample up to 50,000 people with at least one positive historical rating, then ensure the deterministic walkthrough person is included. Sampling uses no later activity. Neighbours are found only in this pool; the result is not the full-population nearest-neighbour solution. The query person needs no three-like minimum. CSR stores observed likes without allocating a dense people-by-attractions matrix.

In [ ]:
# Use a sparse table so we store the observed likes rather than a huge grid of mostly zeros.
from scipy.sparse import csr_matrix


# Fix the random sample and choose the initial neighbour/overlap settings here; they are not proven optimal.
SEED = 20261017
POOL_SIZE = 50000
NEIGHBOURS = 30
MIN_SHARED = 1


# People need at least one historical positive to have a nonempty interaction row.
counts = np.bincount(positive['user'], minlength=NUSER)
eligible = np.flatnonzero(counts > 0)


# Sample the neighbour pool using earlier records only; later outcomes do not choose these people.
rng = np.random.default_rng(SEED)
pool = np.sort(rng.choice(eligible, min(POOL_SIZE, len(eligible)), replace=False))


# Include the fixed example person without increasing the pool size.
if DEMO_USER not in pool:
    pool = np.sort(np.r_[pool[:-1], DEMO_USER])


# Keep only likes from sampled people and map their global user numbers to local matrix rows.
in_pool = np.zeros(NUSER, dtype=bool)
in_pool[pool] = True
hp = positive[in_pool[positive['user']]]
rows = np.searchsorted(pool, hp['user'])


# Store one 1 per distinct person–place like. Columns still use the global attraction row numbers.
interaction = csr_matrix((np.ones(len(hp), dtype=np.float64), (rows, hp['item'])), shape=(len(pool), NITEM))
assert interaction.max() == 1


# Normalize each person’s row; its length is the square root of their number of likes.
row_norm = np.sqrt(np.asarray(interaction.multiply(interaction).sum(axis=1)).ravel())
normalized = interaction.multiply((1 / row_norm)[:, None]).tocsr()


# Locate the example person’s row before comparing them with the pool.
query = int(np.searchsorted(pool, DEMO_USER))


# Count common liked places separately so a high cosine can be checked against actual overlap.
shared = (interaction @ interaction[query].T).toarray().ravel()


# Dot products of normalized rows give user-user cosine; self-similarity is removed immediately.
similarity = (normalized @ normalized[query].T).toarray().ravel()
similarity[query] = 0


# Save exactly who was sampled and the settings needed to reconstruct the same calculation.
np.save(OUT / 'neighbour_pool.npy', pool)
(OUT / 'neighbour_configuration.json').write_text(
    json.dumps(
        dict(
            seed=SEED,
            pool_size=len(pool),
            neighbours=NEIGHBOURS,
            min_shared=MIN_SHARED,
            query_user=DEMO_USER,
        ),
        indent=2,
    ),
)
display(
    pd.Series(
        {
            'people_in_pool': len(pool),
            'attraction_columns': NITEM,
            'stored_likes': interaction.nnz,
            'people_with_shared_likes_excluding_self': int(((shared > 0) & (np.arange(len(pool)) != query)).sum()),
        },
    ).to_frame(
        'count',
    ),
)


# Verify sparse row identity, binary values and deterministic sampling assumptions.
assert len(np.unique(pool)) == len(pool) and np.all(np.diff(pool) > 0)
assert interaction.shape == (len(pool), NITEM)
assert np.all(interaction.data == 1) and (row_norm > 0).all()
assert pool[query] == DEMO_USER
sampling_audit = {
    'historical_positive_users_before_sampling': len(eligible),
    'sampled_people': len(pool),
    'full_positive_pairs': len(positive),
    'sampled_positive_pairs': interaction.nnz,
    'query_included_deliberately': True,
    'sample_seed': SEED,
    'sampling_fraction': len(pool) / len(eligible),
}
(OUT / 'sampling_audit.json').write_text(json.dumps(sampling_audit, indent=2))
display(pd.Series(sampling_audit).to_frame('value'))


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 6. Inspect the neighbours before using their votes

Start with up to 30 neighbours who share at least one liked place. Sort similarity descending, then compact user index. One shared popular place can produce a positive similarity but weak evidence. Display shared counts and actual shared place names, not only a similarity decimal.

In [ ]:
# Return up to the requested neighbour count, using only other people with enough common liked places.
def choose_neighbours(limit=30, min_shared=1):
    if isinstance(limit, bool) or not isinstance(limit, (int, np.integer)) or limit < 0:
        raise ValueError('Neighbour limit must be a nonnegative integer.')
    if isinstance(min_shared, bool) or not isinstance(min_shared, (int, np.integer)) or min_shared < 0:
        raise ValueError('Minimum shared likes must be a nonnegative integer.')

    # Sort most similar first; use the user number only to make exact ties repeatable.
    order = np.lexsort((pool, -similarity))

    # Remove zero overlap and the person themself before applying the neighbour limit.
    return order[(similarity[order] > 0) & (shared[order] >= min_shared) & (order != query)][:limit]


# Apply the initial settings and keep the query’s liked attraction IDs for explanation.
neighbours = choose_neighbours(NEIGHBOURS, MIN_SHARED)
query_items = set(interaction[query].indices)


# Show each neighbour’s similarity beside the shared count and their total history size.
nt = pd.DataFrame(
    {
        'user_index': pool[neighbours],
        'cosine': similarity[neighbours],
        'shared_liked_places': shared[neighbours].astype(int),
        'all_liked_places': np.asarray(interaction[neighbours].sum(axis=1)).ravel().astype(int),
    },
)


# Look up the actual shared place names so we can inspect what made these people look similar.
nt['shared_names'] = ['; '.join(items.nameAttraction.iloc[sorted(query_items & set(interaction[n].indices))]) for n in neighbours]
display(nt)


# Save the neighbour evidence, not just a final recommendation score.
nt.to_csv(OUT / 'real_neighbours.csv', index=False)
print(
    'Positive-overlap people:',
    int(((similarity > 0) & (shared >= MIN_SHARED)).sum()),
    '; retained:',
    len(neighbours),
)


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 7. Turn neighbour likes into candidate scores

For the selected neighbours, compute the similarity-weighted mean of their binary likes. The denominator sums similarities of all selected neighbours, not just supporters of a candidate. Keeping the same denominator preserves differences in endorsement strength. Display the top candidate and each neighbour’s contribution.

In [ ]:
# Return one weighted endorsement score per destination candidate for the supplied neighbour rows.
def vote_scores(ns):
    ns = np.asarray(ns)
    if ns.ndim != 1 or (len(ns) and (ns.dtype.kind not in 'iu' or ns.min() < 0 or ns.max() >= len(pool))):
        raise ValueError('Neighbour rows must be a one-dimensional array of valid integer row numbers.')
    if len(ns) != len(np.unique(ns)):
        raise ValueError('A neighbour must not vote twice.')
    if len(ns) and (query in ns or not (similarity[ns] > 0).all()):
        raise ValueError('Use other people with positive similarity only.')

    # With no neighbours, return zero CF support so the content/popularity routes can take over.
    if len(ns) == 0:
        return np.zeros(len(candidates))

    # Add similarity-weighted likes and divide by all neighbour weights, not only the people who liked each candidate.
    return np.asarray(interaction[ns][:, candidates].T @ similarity[ns]).ravel() / similarity[ns].sum()


# Calculate the real candidate scores, then identify places already reviewed by the query.
cf_scores = vote_scores(neighbours)
seen = np.isin(DEMO_USER * NITEM + candidates, seen_pairs)
ids = items.attractionID.to_numpy()[candidates]


# Sort CF scores first, positive-reviewer counts second and IDs third; keep only unseen positive-support candidates.
rank = np.lexsort((ids, -support[candidates], -cf_scores))


# Sort CF scores first, positive-reviewer counts second and IDs third; keep only unseen positive-support candidates.
rank = rank[~seen[rank] & (cf_scores[rank] > 0)]


# Attach names and count the actual endorsing neighbours; one high score does not tell us the evidence depth.
cf_preview = items.iloc[candidates[rank[:10]]][['attractionID', 'nameAttraction']].copy()
cf_preview['cf_score'] = cf_scores[rank[:10]]
cf_preview['supporting_neighbours'] = [int(interaction[neighbours,
    int(candidates[i])].sum()) for i in rank[:10]]
display(cf_preview)
cf_preview.to_csv(OUT / 'cf_supported_candidates.csv', index=False)


# If there is a supported candidate, explain the first one by showing each neighbour’s contribution.
if len(rank):

    # Read whether each selected neighbour liked this candidate: 1 for an observed like, otherwise 0.
    top = int(rank[0])
    endorse = interaction[neighbours, int(candidates[top])].toarray().ravel()

    # Divide each supporting neighbour’s similarity by the same all-neighbour denominator used in scoring.
    contributions = pd.DataFrame(
        {
            'user_index': pool[neighbours],
            'similarity': similarity[neighbours],
            'liked_candidate': endorse.astype(int),
            'score_contribution': similarity[neighbours] * endorse / similarity[neighbours].sum(),
        },
    )
    display(contributions[contributions.liked_candidate == 1])

    # The individual contributions must add up to the original score.
    assert np.isclose(contributions.score_contribution.sum(), cf_scores[top])
    contributions.to_csv(OUT / 'top_cf_contributions.csv', index=False)


# Save the neighbour rows, similarities, candidate mapping and scores for an independent reconstruction.
np.savez(
    OUT / 'cf_reload.npz',
    pool=pool,
    query=DEMO_USER,
    neighbours=neighbours,
    candidates=candidates,
    similarity=similarity,
    cf_scores=cf_scores,
)


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 8. Build the final list with three clearly labelled sources

Keep positive CF-supported candidates first. Fill any remaining places with positive own-interest category matches, then with destination popularity. Remove reviewed places and duplicates at every stage. Scores from CF and category cosine are different quantities; do not combine them into one unexplained score.

In [ ]:
# Build one list in stages: CF matches, then category matches, then popular fillers. Keep each row’s source.
def hybrid_list(cf, own, seen_mask, capacity=10):
    if np.asarray(seen_mask).dtype.kind != 'b':
        raise ValueError('Reviewed-place flags must be true/false values, not strings or numbers.')

    # Convert inputs to aligned arrays; the two method scores remain separate.
    cf = np.asarray(cf, dtype=float)
    own = np.asarray(own, dtype=float)
    exclude = np.asarray(seen_mask, dtype=bool)

    # Stop if a score or reviewed-item flag refers to the wrong number of candidates.
    if any((x.shape != (len(candidates),) for x in [cf, own, exclude])):
        raise ValueError('All arrays must align with candidates.')

    # Do not rank missing, infinite or negative scores.
    if not np.isfinite(cf).all() or not np.isfinite(own).all() or (cf < 0).any() or (own < 0).any():
        raise ValueError('Use finite nonnegative method scores.')

    # Reject an invalid number of requested positions.
    if isinstance(capacity, bool) or not isinstance(capacity, int) or capacity < 0:
        raise ValueError('Capacity must be a nonnegative integer.')

    # Remember chosen candidate positions and their sources so later stages cannot add duplicates.
    selected = []
    sources = []

    # Try the three sources in order. CF and category stages require positive scores; popularity fills remaining gaps.
    for values, label, positive_only in [
        (cf, 'Similar travellers', True),
        (own, 'Own-interest category match', True),
        (support[candidates].astype(float), 'Destination popularity', False),
    ]:

        # Within this stage sort by its own score, then historical positive-reviewer count and ID.
        ordered = np.lexsort((ids, -support[candidates], -values))

        # Walk down this stage’s ranking until the list is full or eligible choices run out.
        for i in ordered:
            if len(selected) >= capacity:
                break

            # Skip reviewed places, duplicates and unsupported matches before accepting a candidate.
            if exclude[i] or i in selected or (positive_only and values[i] <= 0):
                continue
            selected.append(int(i))
            sources.append(label)

    # Keep the accepted order and turn candidate positions back into attraction rows.
    chosen = np.array(selected, dtype=int)
    result = items.iloc[candidates[chosen]][['attractionID', 'nameAttraction', 'locationAttraction']].copy().reset_index(
        drop=True,
    )

    # Show both method scores; they are different quantities and are not a common probability.
    result['cf_score'] = cf[chosen]
    result['category_cosine'] = own[chosen]
    result['positive_reviewers'] = support[candidates[chosen]]

    # Keep the real source on each row, including category matches used after CF runs out.
    result['source'] = sources
    result['fallback_from_cf'] = result.source != 'Similar travellers'
    return result


# Calculate the content fallback from the same person’s category profile.
own_scores = unit_categories[candidates] @ profile


# Apply the staged policy and show how many positions came from each source.
final = hybrid_list(cf_scores, own_scores, seen, CONFIG['top_k'])
display(final)
final.to_csv(OUT / 'real_hybrid_list.csv', index=False)
display(final.source.value_counts().to_frame('positions'))


# Rank is explicit, and each row exposes evidence from both methods without conflating them.
final.insert(0, 'rank', np.arange(1, len(final) + 1))
final_positions = np.searchsorted(ids, final.attractionID.to_numpy())
# Candidate IDs follow the sorted catalogue order, so searchsorted maps IDs back to candidate positions.
assert np.array_equal(ids[final_positions], final.attractionID.to_numpy())
final['supporting_neighbours'] = [int(interaction[neighbours, candidates[i]].sum()) for i in final_positions]
final['category_labels'] = ['; '.join(np.array(category_names)[categories[candidates[i]] > 0]) for i in final_positions]
final['eligible_city'] = True
final['previously_reviewed'] = False
display(
    final[['rank', 'nameAttraction', 'source', 'cf_score', 'supporting_neighbours', 'category_cosine', 'positive_reviewers']],
)
final.to_csv(OUT / 'real_hybrid_list.csv', index=False)


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 9. Make the 4 + 3 + 3 example explicit

The real list may not require all fallback stages. Separately construct a labelled branch check with four CF-supported positions, three category-only matches and three popularity-only candidates. The imposed scores and availability below are illustrative; they are not the real neighbour result.

In [ ]:
# Find unseen category matches and zero-match places for an explicitly controlled branch example.
positive_own = np.flatnonzero((own_scores > 0) & ~seen)
zero_own = np.flatnonzero((own_scores == 0) & ~seen)
assert len(positive_own) >= 7 and len(zero_own) >= 3


# Leave seven category matches and three zero-match places available; this is an imposed test scenario.
chosen = np.r_[positive_own[:7], zero_own[:3]]
restricted = np.ones(len(candidates), dtype=bool)
restricted[chosen] = False


# Assign made-up CF scores to four of those matches to exercise the 4 + 3 + 3 policy.
imposed = np.zeros(len(candidates))
imposed[positive_own[:4]] = [0.4, 0.3, 0.2, 0.1]


# Build the controlled list and check the source of every position. These scores are not real neighbour evidence.
illustration = hybrid_list(imposed, own_scores, restricted, 10)
display(illustration)
assert illustration.source.tolist() == ['Similar travellers'] * 4 + ['Own-interest category match'] * 3 + ['Destination popularity'] * 3
illustration.to_csv(OUT / 'controlled_4_3_3.csv', index=False)


# Without CF support, category matching takes over; without either signal, popularity takes over.
no_neighbours = hybrid_list(np.zeros(len(candidates)), own_scores, seen, 10)
no_information = hybrid_list(np.zeros(len(candidates)), np.zeros(len(candidates)), seen, 10)
assert 'Similar travellers' not in set(no_neighbours.source)
assert (no_information.source == 'Destination popularity').all()


# Leave only two places available and check that the list stays short rather than duplicating entries.
short_mask = np.ones(len(candidates), dtype=bool)
short_mask[:2] = False
assert len(hybrid_list(cf_scores, own_scores, short_mask, 10)) == 2


# A request for zero positions must return an empty list.
assert hybrid_list(cf_scores, own_scores, seen, 0).empty


# Keep imposed scores explicitly named in the saved controlled example.
illustration['scenario'] = 'CONTROLLED: imposed CF scores and limited availability'
illustration['imposed_cf_score'] = illustration['cf_score']
illustration.to_csv(OUT / 'controlled_4_3_3.csv', index=False)


### What the output tells us

Read the output directly above. This section retains the original method; its quantities are regenerated for the current settings. Use the displayed records and source labels to interpret this run; baseline reproduction and the specific verified claims are reported in the final audit.

## 10. Do neighbour settings change the result?

Compare 10, 30 and 50 neighbours; then require at least two shared likes. These are diagnostics, not validation-selected settings. Keep the sampled pool fixed. Measure supported candidates, fallback positions and shared final top-ten entries. Overlap is not accuracy.

In [ ]:
# Keep the original query, pool and candidate rules fixed for every setting below.
baseline = set(final.attractionID)
checks = []
traces = []
contribution_rows = []
setting_lists = []


# Save every retained neighbour and every positive contribution, not only the top recommendation.
for limit, min_shared in [(10, 1), (30, 1), (50, 1), (30, 2)]:
    ns = choose_neighbours(limit, min_shared)
    values = vote_scores(ns)
    output = hybrid_list(values, own_scores, seen, CONFIG['top_k'])
    label = f'k{limit}_shared{min_shared}'
    eligible_count = int(((similarity > 0) & (shared >= min_shared) & (np.arange(len(pool)) != query)).sum())
    checks.append(
        {
            'configuration': label,
            'max_neighbours': limit,
            'minimum_shared_likes': min_shared,
            'positive_overlap_people_before_threshold': int((similarity > 0).sum()),
            'eligible_neighbours_after_threshold': eligible_count,
            'retained_neighbours': len(ns),
            'eligible_unseen_candidates_all_methods': int((~seen).sum()),
            'unseen_cf_supported_candidates': int(((values > 0) & ~seen).sum()),
            'recommendations_produced': len(output),
            'cf_positions': int((output.source == 'Similar travellers').sum()),
            'category_positions': int((output.source == 'Own-interest category match').sum()),
            'popular_positions': int((output.source == 'Destination popularity').sum()),
            'shared_with_baseline_top10': len(set(output.attractionID) & baseline),
            'ranking_quality_evaluated_here': False,
        },
    )
    for n in ns:
        traces.append(
            {
                'configuration': label,
                'user_index': int(pool[n]),
                'similarity': float(similarity[n]),
                'shared_likes': int(shared[n]),
                'all_likes': int(interaction[n].sum()),
            },
        )
        liked_candidates = np.flatnonzero((interaction[n][:, candidates].toarray().ravel() > 0) & ~seen)
        for i in liked_candidates:
            contribution_rows.append(
                {
                    'configuration': label,
                    'user_index': int(pool[n]),
                    'attractionID': ids[i],
                    'score_contribution': float(similarity[n] / similarity[ns].sum()),
                    'candidate_score': float(values[i]),
                },
            )
    output = output.assign(configuration=label, rank=np.arange(1, len(output) + 1))
    setting_lists.append(output)

sensitivity = pd.DataFrame(checks)
trace_table = pd.DataFrame(traces)
contribution_table = pd.DataFrame(
    contribution_rows,
    columns=['configuration', 'user_index', 'attractionID', 'score_contribution', 'candidate_score'],
)
display(sensitivity)
display(trace_table.query("configuration == 'k30_shared2'"))
sensitivity.to_csv(OUT / 'neighbour_sensitivity.csv', index=False)
trace_table.to_csv(OUT / 'sensitivity_neighbours.csv', index=False)
contribution_table.to_csv(OUT / 'sensitivity_contributions.csv', index=False)
pd.concat(setting_lists, ignore_index=True).to_csv(
    OUT / 'sensitivity_final_lists.csv',
    index=False,
)


# Every supported candidate score must be recoverable from all its retained-neighbour contributions.
if len(contribution_table):
    sums = contribution_table.groupby(['configuration', 'attractionID']).score_contribution.sum()
    originals = contribution_table.groupby(['configuration', 'attractionID']).candidate_score.first()
    assert np.allclose(sums, originals)

baseline_people = set(trace_table.loc[trace_table.configuration == 'k30_shared1', 'user_index'])
strict_people = set(trace_table.loc[trace_table.configuration == 'k30_shared2', 'user_index'])
replacement = trace_table.loc[trace_table.configuration == 'k30_shared2'].copy()
replacement['was_in_initial_30'] = replacement.user_index.isin(baseline_people)
display(replacement)
replacement.to_csv(OUT / 'strict_overlap_member_trace.csv', index=False)
print(
    'Retained from baseline:',
    len(baseline_people & strict_people),
    '; newly admitted:',
    len(strict_people - baseline_people),
)


### What the output tells us

Read the recomputed table, then inspect the member trace. The overlap condition is applied before the top-k cap. Therefore stricter overlap can admit different people, not merely remove voters from the original selected set. The contribution file lists every supported unseen candidate for each setting. More supported candidates or a changed list do not establish better ranking quality.

## 11. Compare complete policies on 2018 recorded likes

This adds an actual **retrospective development comparison**, not parameter selection or a final test. Freeze pre-2018 histories, the original sampled pool and candidate rules. Evaluate the same pooled people who have at least one eligible, previously unreviewed earliest-valid 4/5 record published in 2018. People without such labels have no ranking denominator; exclude them from ranking means and count them explicitly. No-history visitors are not represented by this historical-positive pool.

Compare popularity, category→popularity and CF→category→popularity. Defaults stay at 30 neighbours/minimum one shared like. Rank every eligible candidate, not sampled negatives. Recall@10 = retrieved eligible positives / all eligible positives. NDCG@10 uses binary gains, discount 1/log2(rank+1), and ideal length min(10, positive count). Average users equally. Report pool size, later-label availability, source coverage among the evaluable people and output composition. Do not generalize these percentages to all visitors.

The supplied category snapshot has no annual versions. This comparison measures agreement with later recorded likes under fixed supplied metadata; it does not reconstruct all information that would have been available historically. Previously inspected 2018/2019 results prevent an untouched-test claim. The new comparison uses 2018 only; it does not tune settings or inspect 2019 results.

In [ ]:
# Read the design saved before labels were inspected. Existing defaults remain fixed.
assert protocol['cf_defaults'] == {'neighbours': NEIGHBOURS, 'minimum_shared': MIN_SHARED}
EVAL_YEAR = 2018
end_day = int(np.datetime64('2019-01-01', 'D').astype(np.int64))


# Later earliest-valid likes are labels only. Neither these rows nor their popularity enter earlier inputs.
year_positive = dedup[(dedup['day'] >= cut) & (dedup['day'] < end_day) & (dedup['rating'] >= 4)]
pool_year_positive = year_positive[in_pool[year_positive['user']]]
label_candidate = candidate_mask[pool_year_positive['item']]
label_keys = pool_year_positive['user'].astype(np.int64) * NITEM + pool_year_positive['item']
label_unseen = ~np.isin(label_keys, seen_pairs)
labels = pool_year_positive[label_candidate & label_unseen]
eval_users = np.unique(labels['user'])
flow = {
    'training_pool_people': len(pool),
    'pool_people_with_any_2018_positive': len(np.unique(pool_year_positive['user'])),
    'pool_2018_positive_pairs': len(pool_year_positive),
    'targets_excluded_by_candidate_rules': int((~label_candidate).sum()),
    'targets_excluded_as_seen_after_candidate_rules': int((label_candidate & ~label_unseen).sum()),
    'eligible_unseen_target_pairs': len(labels),
    'evaluable_people': len(eval_users),
    'pool_people_without_eligible_labels': len(pool) - len(eval_users),
}
display(pd.Series(flow).to_frame('count'))
(OUT / 'evaluation_population.json').write_text(json.dumps(flow, indent=2))
assert len(eval_users) > 0, 'No eligible later positives: ranking evaluation is incomplete, not a zero-quality result.'


# Reuse exactly the same sparse cosine and tie rules for any query row.
def find_for_person(row, limit=30, min_shared=1):

    # Count common earlier likes independently of the cosine score.
    common = (interaction @ interaction[row].T).toarray().ravel()
    sims = (normalized @ normalized[row].T).toarray().ravel()
    sims[row] = 0

    # Remove self and insufficient overlap before applying the neighbour cap.
    ordered = np.lexsort((pool, -sims))
    selected = ordered[(ordered != row) & (sims[ordered] > 0) & (common[ordered] >= min_shared)][:limit]
    values = (
        (
        np.asarray(interaction[selected][:, candidates].T @ sims[selected]).ravel() / sims[selected].sum()
        if len(selected)
        else np.zeros(len(candidates))
    )
    )
    return selected, values

assert np.array_equal(find_for_person(query, NEIGHBOURS, MIN_SHARED)[0], neighbours)
assert np.array_equal(find_for_person(query, NEIGHBOURS, MIN_SHARED)[1], cf_scores)


def ranking_metrics(ranked_ids, positives, k):
    if not positives:
        raise ValueError('No target denominator: count this person separately, not as zero recall.')

    # Mark known later positives, then discount hits that appear farther down the list.
    hits = np.array([value in positives for value in ranked_ids[:k]], dtype=float)
    dcg = float(np.sum(hits / np.log2(np.arange(len(hits)) + 2)))

    # Normalize by the best ranking possible for this person’s number of target positives.
    ideal = float(np.sum(1 / np.log2(np.arange(min(k, len(positives))) + 2)))
    return float(hits.sum() / len(positives)), dcg / ideal


# A sparse profile table lets category fallback use prior positives without rescanning all users for every query.
profile_matrix = interaction @ unit_categories
profile_norms = np.linalg.norm(profile_matrix, axis=1)
profile_matrix = profile_matrix / np.maximum(profile_norms[:, None], 1e-12)
assert np.allclose(profile_matrix[query], profile)

metric_rows = []
ranking_rows = []
score_rows = []
np.save(OUT / 'evaluation_label_records.npy', labels)
pd.DataFrame(
    {'user_index': labels['user'], 'attractionID': items.attractionID.to_numpy()[labels['item']]},
).to_csv(
    OUT / 'evaluation_targets.csv',
    index=False,
)

# seen_pairs is already sorted by np.unique. Binary search avoids rescanning millions of keys for every person.
def reviewed_for_person(person):
    keys = int(person) * NITEM + candidates
    positions = np.searchsorted(seen_pairs, keys)
    valid_positions = positions < len(seen_pairs)
    reviewed = np.zeros(len(candidates), dtype=bool)
    reviewed[valid_positions] = seen_pairs[positions[valid_positions]] == keys[valid_positions]
    return reviewed


# Compare the faster lookup with the original expression on the walkthrough and cohort endpoints.
for check_person in [DEMO_USER, int(eval_users[0]), int(eval_users[-1])]:
    assert np.array_equal(reviewed_for_person(check_person), np.isin(int(check_person) * NITEM + candidates, seen_pairs))


for person in eval_users:

    # Use only this person’s earlier profile and reviewed places; later rows remain labels.
    row = int(np.searchsorted(pool, person))
    already_reviewed = reviewed_for_person(person)
    selected, votes = find_for_person(row, NEIGHBOURS, MIN_SHARED)
    own = unit_categories[candidates] @ profile_matrix[row]
    targets = set(items.attractionID.to_numpy()[labels['item'][labels['user'] == person]])
    score_rows.append((int(person), votes, own, already_reviewed))

    # Compare all three complete policies on the same person, candidate set and target set.
    policy_inputs = [
        ('popularity', np.zeros(len(candidates)), np.zeros(len(candidates))),
        ('category_then_popularity', np.zeros(len(candidates)), own),
        ('cf_then_category_then_popularity', votes, own),
    ]
    for policy_name, policy_cf, policy_own in policy_inputs:
        output = hybrid_list(policy_cf, policy_own, already_reviewed, CONFIG['top_k'])
        recall, ndcg = ranking_metrics(output.attractionID.tolist(), targets, CONFIG['top_k'])
        metric_rows.append(
            {
                'user_index': int(person),
                'policy': policy_name,
                'eligible_targets': len(targets),
                'historical_likes': int(counts[person]),
                'recall_at_10': recall,
                'ndcg_at_10': ndcg,
                'list_length': len(output),
                'selected_neighbours': len(selected),
                'cf_positions': int((output.source == 'Similar travellers').sum()),
                'category_positions': int((output.source == 'Own-interest category match').sum()),
                'popular_positions': int((output.source == 'Destination popularity').sum()),
            },
        )
        for rank_number, item in enumerate(output.itertuples(), 1):
            ranking_rows.append(
                {
                    'user_index': int(person),
                    'policy': policy_name,
                    'rank': rank_number,
                    'attractionID': item.attractionID,
                    'source': item.source,
                    'is_recorded_positive': item.attractionID in targets,
                },
            )


# Keep per-person values before taking an equal-weight mean across people.
metrics = pd.DataFrame(metric_rows)
rankings = pd.DataFrame(ranking_rows)
assert metrics.groupby('policy').user_index.nunique().nunique() == 1
assert not rankings.duplicated(['user_index', 'policy', 'attractionID']).any()
quality = metrics.groupby('policy').agg(
    people=('user_index', 'size'),
    recall_at_10=('recall_at_10', 'mean'),
    ndcg_at_10=('ndcg_at_10', 'mean'),
    mean_list_length=('list_length', 'mean'),
    mean_cf_positions=('cf_positions', 'mean'),
    mean_category_positions=('category_positions', 'mean'),
    mean_popular_positions=('popular_positions', 'mean'),
)
quality['catalogue_coverage'] = rankings.groupby('policy').attractionID.nunique() / len(candidates)
quality['any_cf_share_of_people'] = metrics.assign(any_cf=metrics.cf_positions > 0).groupby('policy').any_cf.mean()
quality['all_popular_share_of_people'] = metrics.assign(all_popular=metrics.popular_positions == metrics.list_length).groupby('policy').all_popular.mean()
metrics['history_group'] = pd.cut(metrics.historical_likes, [0, 2, 5, 10, np.inf], labels=['1-2', '3-5', '6-10', '11+'])

# Show small history-group counts beside their means instead of hiding them in one average.
groups = metrics.groupby(['policy', 'history_group'], observed=True).agg(
    people=('user_index', 'size'),
    recall_at_10=('recall_at_10', 'mean'),
    ndcg_at_10=('ndcg_at_10', 'mean'),
    mean_cf_positions=('cf_positions', 'mean'),
)
display(quality)
display(groups)
metrics.to_csv(OUT / 'policy_per_user_metrics.csv', index=False)
rankings.to_csv(OUT / 'policy_rankings.csv', index=False)
quality.to_csv(OUT / 'policy_summary.csv')
groups.to_csv(OUT / 'policy_history_groups.csv')
np.savez_compressed(
    OUT / 'evaluation_scores.npz',
    users=np.array([r[0] for r in score_rows]),
    cf=np.array([r[1] for r in score_rows]),
    category=np.array([r[2] for r in score_rows]),
    seen=np.array([r[3] for r in score_rows]),
    candidates=candidates,
)
np.save(OUT / 'historical_positive_counts.npy', counts)
print(
    'Macro means use the same later-positive people. No setting is selected and no causal visitor effect is measured.',
)


### What the output tells us

All three policies were evaluated on the same 209 people with 424 eligible 2018 positive targets. Each person contributes equal weight to the mean:

| Policy | Recall@10 | NDCG@10 | Catalogue coverage |
|---|---:|---:|---:|
| Popularity | 0.2979 | 0.1933 | 1.19% |
| Category → popularity | 0.1612 | 0.1016 | 13.20% |
| CF → category → popularity | 0.1968 | 0.1244 | 15.92% |

These are observed development-sample means. CF plus fallback matches more recorded positives than category matching here, but neither exceeds popularity. The CF policy reaches more distinct places than popularity; coverage is a different property, not a reason to declare it the better ranking.

104 of the 209 people receive at least one CF-sourced recommendation. Across these evaluable people, the hybrid list averages 2.79 CF positions and 7.21 category positions; it needs no popularity fillers in this sample. These percentages do not describe all 50,000 pooled people or new visitors. The 49,791 pooled people without eligible later labels have no ranking denominator and are reported separately.

No setting was selected from these results. The comparison uses supplied snapshot categories and is not a historically exact reconstruction, an untouched test, or evidence that recommendations caused visits. Small history groups should be read with their displayed sample sizes. A final selection/evaluation protocol and actual visitor-response study remain future work.

## 12. Check the handoff

Verify shared counts, self-exclusion, neighbour score arithmetic, unique unseen results, correct source order and controlled fallback behaviour. Save the interaction matrix and score arrays so another process can reconstruct all CF candidate scores. No browser, audio or online-impact validation is claimed.

In [ ]:
from scipy.sparse import save_npz


# Save observed likes in sparse form so another process can reconstruct the candidate scores.
save_npz(OUT / 'interaction.npz', interaction)


# Check that the query did not vote for themself and every neighbour has positive similarity.
assert query not in neighbours and (similarity[neighbours] > 0).all()


# Recheck cosine using the common-like count divided by the two history lengths.
assert np.allclose(similarity[neighbours], shared[neighbours] / (row_norm[neighbours] * row_norm[query]))


# Check that the scores are real numbers in the expected range, not NaN or infinity.
assert np.isfinite(cf_scores).all() and (cf_scores >= 0).all() and (cf_scores <= 1 + 1e-12).all()


# Reject duplicates, too many results or places already reviewed by this person.
assert final.attractionID.is_unique and len(final) <= 10 and (not np.isin(final.attractionID,
    ids[seen]).any())


# Every row labelled as a CF recommendation must have a genuinely positive CF score.
lookup = dict(zip(ids, cf_scores))
assert all((lookup[i] > 0 for i in final.loc[final.source == 'Similar travellers', 'attractionID']))


# Check that the final list follows the declared CF → category → popularity stage order.
source_order = {'Similar travellers': 0, 'Own-interest category match': 1, 'Destination popularity': 2}
assert final.source.map(source_order).tolist() == sorted(final.source.map(source_order).tolist())


# Save the observed findings with their scope, rather than presenting calculation checks as evidence of business impact.
summary = {
    'query_user': DEMO_USER,
    'pool_people': len(pool),
    'positive_overlap_people': int((similarity > 0).sum()),
    'selected_neighbours': len(neighbours),
    'top_cf_name': str(cf_preview.iloc[0].nameAttraction) if len(cf_preview) else None,
    'top_cf_score': float(cf_preview.iloc[0].cf_score) if len(cf_preview) else None,
    'cf_supported_unseen': len(rank),
    'source_counts': {k: int(v) for k, v in final.source.value_counts().items()},
    'checks': 'passed',
    'scope': 'user-based CF and hybrid policy mechanisms; not a new ranking-quality or business-impact evaluation',
}


# Write a short record of this run’s result and show it below.
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
display(pd.Series(summary).to_frame('result'))


# Check failures deliberately: bad inputs should be rejected, not quietly change the neighbour list.
def must_reject(action):
    try:
        action()
    except (ValueError, TypeError):
        return
    raise AssertionError('Invalid input was silently accepted.')

must_reject(lambda: choose_neighbours(-1, 1))
must_reject(lambda: choose_neighbours(True, 1))
must_reject(lambda: vote_scores(np.array([query])))
must_reject(lambda: vote_scores(np.array([neighbours[0], neighbours[0]])))
must_reject(lambda: hybrid_list(cf_scores[:-1], own_scores, seen, 10))
must_reject(lambda: hybrid_list(np.full(len(candidates), np.nan), own_scores, seen, 10))
assert len(choose_neighbours(0, 1)) == 0
assert np.array_equal(vote_scores(np.array([], dtype=int)), np.zeros(len(candidates)))
assert hybrid_list(cf_scores, own_scores, np.ones(len(candidates), dtype=bool), 10).empty


# Hand-check the metric formulas, including no-hit, perfect, later-hit and no-denominator cases.
assert ranking_metrics(['A', 'B', 'C'], {'A', 'B'}, 3) == (1.0, 1.0)
assert ranking_metrics(['C', 'A', 'B'], {'A', 'B'}, 3)[1] < 1
assert ranking_metrics(['C'], {'A'}, 3) == (0.0, 0.0)
must_reject(lambda: ranking_metrics(['A'], set(), 3))


# A zero-history row has no mathematical cosine. Use explicit routing behavior in a controlled sparse example.
zero_example = csr_matrix(np.array([[0, 0], [1, 0], [1, 1]], dtype=float))
z_norm = np.sqrt(np.asarray(zero_example.multiply(zero_example).sum(axis=1)).ravel())
z_unit = zero_example.multiply((1 / np.maximum(z_norm, 1))[:, None]).tocsr()
z_sim = (z_unit @ z_unit.T).toarray()
assert np.isfinite(z_sim).all() and np.array_equal(z_sim, z_sim.T)
assert (z_sim[0] == 0).all() and np.isclose(z_sim[1, 2], 1 / np.sqrt(2))


# Save the core arrays needed to reconstruct scores and enforce that original source files were not changed.
np.savez(
    OUT / 'audited_core.npz',
    candidates=candidates,
    seen=seen,
    profile=profile,
    cf_scores=cf_scores,
    pool=pool,
    query_row=query,
    similarity=similarity,
    shared=shared,
    neighbours=neighbours,
)
summary['development_evaluation_people'] = len(eval_users)
summary['development_evaluation_year'] = EVAL_YEAR
summary['runtime_seconds'] = time.perf_counter() - AUDIT_STARTED
summary['verification_scope'] = 'fresh-kernel mechanisms + fixed 2018 retrospective development comparison; not a historical-exact or final test'
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
display(pd.Series(summary).to_frame('audited_run_result'))


# An empty candidate catalogue is different from a full catalogue whose items are all excluded.
# Temporarily replace only the candidate mapping for this controlled check, then restore it immediately.
saved_candidates, saved_ids = candidates, ids
try:
    candidates = np.array([], dtype=int)
    ids = np.array([], dtype=str)
    assert hybrid_list(np.array([]), np.array([]), np.array([], dtype=bool), 10).empty
finally:
    candidates, ids = saved_candidates, saved_ids

must_reject(lambda: hybrid_list(cf_scores, own_scores, np.array(['False'] * len(candidates)), 10))


# Verify that overlap filtering happens before the neighbour cap, rather than after it.
saved_pool, saved_similarity, saved_shared, saved_query = pool, similarity, shared, query
try:
    pool = np.array([10, 20, 30, 40])
    similarity = np.array([0.9, 0.8, 0.7, 0.0])
    shared = np.array([1, 2, 2, 0])
    query = 3
    assert choose_neighbours(limit=1, min_shared=2).tolist() == [1]
finally:
    pool, similarity, shared, query = saved_pool, saved_similarity, saved_shared, saved_query


# State the new comparison scope accurately; the original demonstration fields above remain comparable.
summary['scope'] = 'user-CF demonstration and fixed 2018 retrospective development comparison'
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))


### What the output tells us

The run verifies the implemented historical snapshot mechanism, input guards, score reconstruction inputs, metric counterexamples and a common 2018 development comparison. A separate process will check score arithmetic and metrics from saved evidence. Limitations remain: unversioned metadata, sampled users with prior likes, incomplete feedback, no production or causal evidence, and no final untouched evaluation.